# Explainability and decision analysis

Inspect validation perturbations, frozen test reliability, local single-feature explanations, and paired simulation economics.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

root = Path.cwd()
if not (root / "deployment").exists():
    root = root.parent
snapshot = root / "deployment/snapshot"
profile = json.loads((snapshot / "data_report.json").read_text())
model = json.loads((snapshot / "model.json").read_text())
def sql(name):
    return pd.read_csv(snapshot / "analytics" / f"{name}.csv")


In [2]:
display(pd.DataFrame(model["importance"]))
display(pd.DataFrame(model["reliability"]))
scores = pd.read_parquet(snapshot/"scores.parquet")
display(scores.nlargest(5,"expected_exposure")[["claim_id","risk","conditional_amount","conditional_low","conditional_high","reason"]])
display(pd.DataFrame(model["segments"]).query("dimension == 'Quarter'")[["group","n","prevalence","ap","brier"]])


,feature,ap_drop
0,workload,0.032910
1,liability,0.009433
2,complexity,0.009356
3,severity,0.001021
4,report_delay,0.000510
5,documented,0.000487
6,early_events,0.000487
7,coverage,0.000164
8,vendor_supp,0.000155
9,reserve,0.000139


,bin,mean_score,actual_rate,n
0,0,0.056573,0.059817,23221
1,1,0.107269,0.107330,5348
2,2,0.222326,0.166078,566


,claim_id,risk,conditional_amount,conditional_low,conditional_high,reason
4905,94694,0.211538,1821.418818,1343.977447,2298.860188,workload
5540,95360,0.211538,1788.949765,1311.508394,2266.391135,complexity
5354,95160,0.211538,1459.337721,981.896351,1936.779092,workload
3660,93394,0.117391,2542.045217,2064.603846,3019.486587,complexity
5288,95093,0.117391,2353.511061,1876.069691,2830.952432,complexity


,group,n,prevalence,ap,brier
3,2024Q1,6936,0.069493,0.097243,0.064208
4,2024Q2,6896,0.067140,0.094447,0.062080
5,2024Q3,8146,0.080285,0.125221,0.072523
6,2024Q4,6931,0.064060,0.084617,0.059664
7,2025Q1,226,0.061947,0.089855,0.058339


In [3]:
from src.simulation import summarize
runs = pd.read_csv(snapshot/"scenarios.csv")
display(summarize(runs.query("scenario == 'Normal'")).round(2))
display(runs.groupby(["scenario","strategy"]).net.mean().unstack().round(0))
assert (runs.net - runs.gross + runs.review_expense + runs.operational_harm).abs().max() < 1e-8


,reviewed,detected,gross,expense,net,net_sd,downside,hours,delay
strategy,,,,,,,,,
No additional review,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
Random review,364.00,24.33,278.58,9117.07,-9721.21,621.09,-10383.10,183.90,73.56
Highest amount,364.00,34.33,3780.61,8902.73,-5992.36,699.43,-6754.38,181.30,72.52
Operational rules,364.00,36.33,1457.86,4864.47,-4019.09,188.77,-4170.76,127.60,51.04
Highest risk,364.00,38.67,2161.07,5424.87,-3906.51,573.72,-4561.32,133.90,53.56
Expected net benefit,30.67,2.00,0.00,407.87,-459.39,69.19,-539.28,10.73,4.29


strategy,Expected net benefit,Highest amount,Highest risk,No additional review,Operational rules,Random review
scenario,,,,,,
Delayed documentation,-461.0,-5678.0,-4345.0,0.0,-4819.0,-8915.0
High claim volume,25.0,-3812.0,-3370.0,0.0,-2144.0,-8665.0
Higher investigation costs,-66.0,-8713.0,-8715.0,0.0,-8859.0,-11410.0
Higher leakage prevalence,-226.0,-5250.0,-2315.0,0.0,-3081.0,-8535.0
Limited adjuster capacity,-379.0,-3730.0,-2388.0,0.0,-1940.0,-4907.0
Lower effectiveness,0.0,-7496.0,-4976.0,0.0,-4767.0,-9721.0
Normal,-459.0,-5992.0,-3907.0,0.0,-4019.0,-9721.0
Vendor price inflation,-651.0,-5047.0,-3366.0,0.0,-3655.0,-9652.0


Local perturbations hold the other inputs fixed and can form combinations uncommon in the training data. The simulated policy uses planning assumptions for response, not intervention effects learned from observed claims. Its replay value depends on capacity, eligibility, actual costs, and mechanism match.